In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType
from pyspark.ml.regression import LinearRegression


# types
from pyspark.sql.types import BooleanType
from pyspark.sql.types import DoubleType
from pyspark.sql.types import StringType
from pyspark.sql.types import IntegerType
from pyspark.sql.types import StructField

from pyspark.ml.evaluation import RegressionEvaluator

# visualization
import seaborn as sns
import matplotlib.pyplot as plt

# idk
from pyspark.ml.feature import VectorAssembler

In [12]:
LABEL_COLUMN='price'
SEED=42

spark = SparkSession.builder \
    .appName("Airbnb") \
    .getOrCreate()

raw_df = (
    spark.read
    .option("header", "true")
    .option("sep", ",")
    .option("multiLine", "true")
    .option("quote", "\"")
    .option("escape", "\"")
    .option("mode", "PERMISSIVE")
    .parquet("./total_data.parquet")
)

raw_df.createOrReplaceTempView('raw_listing')

In [13]:
selected_df = spark.sql("""
    SELECT
        CAST(host_is_superhost AS INT) AS host_is_superhost,
        CAST(calculated_host_listings_count AS INT) AS host_listings_count,
        latitude,
        longitude,
        CAST(accommodates AS INT) AS accommodates,
        bathrooms,
        bedrooms,
        beds,
        price,
        CAST(snapshot_month AS INT) AS month
    FROM raw_listing
""")

selected_df.createOrReplaceTempView("transactions_selected")
selected_df.printSchema()
selected_df.show()


root
 |-- host_is_superhost: integer (nullable = true)
 |-- host_listings_count: integer (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- accommodates: integer (nullable = true)
 |-- bathrooms: double (nullable = true)
 |-- bedrooms: double (nullable = true)
 |-- beds: double (nullable = true)
 |-- price: double (nullable = true)
 |-- month: integer (nullable = true)

+-----------------+-------------------+-------------------+-------------------+------------+---------+--------+----+------+-----+
|host_is_superhost|host_listings_count|           latitude|          longitude|accommodates|bathrooms|bedrooms|beds| price|month|
+-----------------+-------------------+-------------------+-------------------+------------+---------+--------+----+------+-----+
|                0|                  6|-22.992810000000002|          -43.43523|          10|      5.0|     3.0|10.0|1100.0|   11|
|                0|                  1|-23.00484254068

## Análise Exploratória

In [14]:
# selected_df_pandas = selected_df.toPandas()

In [ ]:
# these are some helper functions to plot the charts
def get_max_fence(column):
    qt = selected_df.approxQuantile(column, [0.25,0.75], 0.01)

    q1 = qt[0]
    upper = qt[1]

    iqr = upper - q1

    max_fence = upper + 1.5* iqr
    return max_fence

# def box_plot(column):
#     fig, (ax1, ax2) = plt.subplots(1,2)
#     fig.set_size_inches(16,6)
    
#     pdf = selected_df_pandas.select(column)
    
#     _ = sns.boxplot(x=pdf[column], ax=ax1)
#     ax1.set_title(f'{column} boxplot')
    
#     _ = sns.boxplot(x=pdf[column], ax=ax2) 
    
#     ax2.set_title(f'Zooming in the {column} boxplot')
    
#     ax2.set_xlim((-0.1, 1.1 * get_max_fence(column)))

In [21]:


max_price_fence = get_max_fence('price')
max_host_listing = get_max_fence('host_listings_count')
max_beds = get_max_fence('beds')

# box_plot('price')
selected_df = selected_df.dropna()

selected_df = selected_df.filter(col('price') <= max_price_fence)
# selected_df = selected_df.filter(col('host_listings_count') <= max_host_listing)
selected_df = selected_df.filter(col('beds') <= max_beds)

In [22]:
from pyspark.ml.regression import DecisionTreeRegressor
models = [
    (
        "Linear Regression",
        LinearRegression(
            labelCol=LABEL_COLUMN,
            featuresCol="features",
            maxIter=50,
            regParam=0.1,
            elasticNetParam=0.0
        )
    ),
    (
        "Decision Tree Regressor",
        DecisionTreeRegressor(
            labelCol=LABEL_COLUMN,
            featuresCol="features"
        )
    )
]

In [23]:

feature_cols = [
    "host_is_superhost",
    "host_listings_count",
    "latitude",
    "longitude",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "month"
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
prepared_df = assembler.transform(selected_df)

train_prepared, test_prepared = prepared_df.randomSplit([0.8, 0.2], seed=SEED)

print(f"Linhas no Treino: {train_prepared.count()}")
print(f"Linhas no Teste: {test_prepared.count()}")


Linhas no Treino: 528917


Linhas no Teste: 132629


In [24]:

def evaluate_model(model_name: str, predictions) -> None:
    predictions = predictions.cache()
    predictions.createOrReplaceTempView("predictions")

    rmse = RegressionEvaluator(
        labelCol=LABEL_COLUMN,
        predictionCol="prediction",
        metricName="rmse",
    ).evaluate(predictions)
    mae = RegressionEvaluator(
        labelCol=LABEL_COLUMN,
        predictionCol="prediction",
        metricName="mae",
    ).evaluate(predictions)
    r2 = RegressionEvaluator(
        labelCol=LABEL_COLUMN,
        predictionCol="prediction",
        metricName="r2",
    ).evaluate(predictions)

    print(f"RMSE: {rmse:.2f}")
    print(f"MAE: {mae:.2f}")
    print(f"R2: {r2:.6f}")

    predictions.unpersist()

In [25]:
for model_name, estimator in models:
    model = estimator.fit(train_prepared)
    predictions = model.transform(test_prepared)
    evaluate_model(model_name, predictions)

RMSE: 209.74
MAE: 158.52
R2: 0.250398


RMSE: 204.69
MAE: 153.14
R2: 0.286093
